"We chose to recruit 100 participants to achieve sufficient power to at least calculate an overall association between the constructs; a priori power analysis for Pearson’s 𝑟 indicated that a sample of 𝑁 =85 would be sufficient to detect a medium-sized association (𝑟=0.30) with 80% power in a two-tailed test at 𝛼=0.05."

In [63]:
import math
import pingouin as pg

required_n = pg.power_corr(
    r=0.30,                 # assumed true correlation
    power=0.80,             # desired power
    alpha=0.05,             # significance level
    alternative="two-sided"
)

print(f"Unrounded sample size: {required_n:.2f}")
print(f"Required sample size: {math.ceil(required_n)}")

Unrounded sample size: 84.07
Required sample size: 85


In [64]:
import pandas as pd

df = pd.read_csv("data.csv")
df.head()

# SCS1 = Computer Science Achievement
# ICAR16 = General Reasoning Skills
# Essay = Written Communication Skills


,ID,SCS1,ICAR16,Task_Decontextualized,Task_Replication,Task_FeatureAddition,VibesSum,Essay,LLMExperience,QuizFirst
0,1,0.83,0.56,0.42,0.56,0.67,0.55,0.67,5.0,True
1,2,0.50,0.50,0.50,0.56,0.50,0.52,0.65,4.0,True
2,3,0.08,0.31,0.00,0.00,0.00,0.00,0.65,5.0,True
3,4,0.33,0.56,0.00,0.33,0.50,0.28,0.76,5.0,True
4,5,0.50,0.50,0.58,0.67,0.50,0.58,0.82,4.0,False


**Table 3** Correlation matrices between CS achievement (CS), cognitive ability (Cog.), writing skills (Writing), and vibe coding performance (Vibe). All values are Pearson’s 𝑟 with two-tailed p-values in parentheses

In [65]:
from scipy.stats import pearsonr
columns = ["SCS1", "ICAR16", "Essay","VibesSum"]
data = df[columns]


correlation_matrix = data.corr(method="pearson")

pvalue_matrix = pd.DataFrame(
    [[pearsonr(data[x], data[y]).pvalue for y in columns]
     for x in columns],
    index=columns,
    columns=columns
)

print("Correlations:")
print(correlation_matrix.round(3))

print("\nP-values:")
print(pvalue_matrix.round(3))

Correlations:
           SCS1  ICAR16  Essay  VibesSum
SCS1      1.000   0.417  0.126     0.386
ICAR16    0.417   1.000  0.365     0.352
Essay     0.126   0.365  1.000     0.290
VibesSum  0.386   0.352  0.290     1.000

P-values:
           SCS1  ICAR16  Essay  VibesSum
SCS1      0.000     0.0  0.213     0.000
ICAR16    0.000     0.0  0.000     0.000
Essay     0.213     0.0  0.000     0.003
VibesSum  0.000     0.0  0.003     0.000


**RQ2:** Overall association and independence from general
reasoning. (a) Are written-communication skills positively
correlated with vibe coding performance?

In [66]:

result = pearsonr(df["Essay"], df["VibesSum"])
ci = result.confidence_interval(confidence_level=0.95)

print(f'r={result.statistic:.3f}, p= {result.pvalue:.3f}, ci=[{ci[0]:.3f}, {ci[1]:.3f}]')

r=0.290, p= 0.003, ci=[0.099, 0.460]


**For RQ3**, we asked how the predictive value of CS achievement and written-communication skills compare on vibe-coding performance. We compared the unique variance in aggregate vibe-coding accuracy explained by written-communication proficiency and CS achievement using hierarchical OLS.


Using both predictors in
the final model, standardized coefficients indicated positive associations
for both: Writing had the parameter 𝛽 = 0.244 [95% CI
0.063, 0.425], 𝑝 = 0.009; and CS had the parameter 𝛽 = 0.356
[95% CI 0.176, 0.537], 𝑝 < 0.001. Descriptively, CS achievement
is the stronger predictor of aggregate vibe-coding accuracy, but
written-communication skills also show a reliable unique contribution
beyond CS achievement.

In [67]:
import statsmodels.api as sm

from scipy.stats import zscore

columns = ["SCS1", "Essay", "VibesSum"]

df[columns] = df[columns].apply(zscore)


X = df[["SCS1", "Essay"]]
X = sm.add_constant(X)
y = df["VibesSum"]

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:               VibesSum   R-squared:                       0.208
Model:                            OLS   Adj. R-squared:                  0.192
Method:                 Least Squares   F-statistic:                     12.77
Date:                Mon, 07 Sep 2026   Prob (F-statistic):           1.19e-05
Time:                        05:39:44   Log-Likelihood:                -130.20
No. Observations:                 100   AIC:                             266.4
Df Residuals:                      97   BIC:                             274.2
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       1.527e-16      0.090   1.69e-15      1.0